# H3 · Durability and the last-mover advantage (*Zero to One*, ch. 5)

> "A great business is defined by its ability to generate cash flows in the future." … "Moving first is a tactic, not a goal. What really matters is generating cash flows in the future, so being the first mover doesn't do you any good if someone else comes along and unseats you. It's much better to be the last mover."

Thiel's durability question asks: will your market position be defensible 10 and 20 years into the future? He also says that most of a tech company's value lies 10–15 years or more away (about 75% of PayPal's value in 2001 came from cash flows in 2011 and later).

| Claim | Test | Data |
|---|---|---|
| **H3a.** High profitability is durable rather than competed away. | Do firms in the top decile of return on assets stay there 1–10 years later? How fast does above-average profitability revert to the mean (as in Fama & French, 2000)? Compared for base years before and after 2014. | SEC XBRL, 2011–2025 |
| **H3b.** Industry leaders stay leaders (the last mover keeps the market). | The share of industry leaders, and of firms with half their market or more, that still lead five years later, by base year 2011–2020. | SEC XBRL + SIC |
| **H3c.** The giants are durable. | Of the 20 largest US firms by public float in 2014, how many are still in the top 20? Five-year top-20 survival by base year. | SEC public float (the H1 panel) |
| **H3d.** Value lies far in the future. | How much of each large firm's 2014 value had its 2015–2025 free cash flow already repaid? A low share means the price rested on cash flows after 2025, as Thiel says it should for tech firms. | SEC cash-flow statements, public float |

**Caveats**
* Fifteen years of data limit the 10-year tests to base years 2011–2015.
* Firms that stop filing were acquired or failed, and we cannot tell which. They are reported as a separate outcome, never counted as "stayed on top".
* Public float excludes insider stakes, so it understates firm value where founders hold much of the stock, which overstates the payback ratio for those firms.
* Free cash flow here is operating cash flow minus capital expenditure. It ignores acquisitions, and it treats stock-based compensation as costless (the cash-flow statement adds it back).

In [ ]:
import warnings; warnings.filterwarnings("ignore")
import json, numpy as np, pandas as pd, matplotlib.pyplot as plt
import sys; from pathlib import Path  # thiel_common.py lives in the repo root
sys.path.insert(0, str(next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "thiel_common.py").exists())))
import thiel_common as tc
tc.style()
print("synthetic mode" if tc.SYNTHETIC else "live data")

fin = tc.financial_panel(range(2011, 2026))
fin = fin[fin.assets >= 1e7].copy()
print(f"{len(fin):,} firm-years, {fin.cik.nunique():,} firms, {fin.year.min()}-{fin.year.max()}")

## H3a · Does high profitability last?

* **Decile persistence:** of firms in the top decile of ROA in year *t*, the share that are still in the top decile *k* years later, the share that dropped below the median, and the share no longer in the data.
* **Mean reversion:** the slope of a firm's ROA at *t+k*, measured relative to that year's median, on its ROA at *t*, also relative to the median. A slope of 1 means profitability gaps never close; 0 means they vanish entirely. Fama & French (2000) estimate that about 38% of the gap closes each year.

In [ ]:
fin["roa_dev"] = fin.roa - fin.groupby("year").roa.transform("median")
fin["roa_dec"] = fin.groupby("year").roa.transform(lambda s: pd.qcut(s.rank(method="first"), 10, labels=False))
roa = fin.pivot_table(index="cik", columns="year", values="roa_dev")
dec = fin.pivot_table(index="cik", columns="year", values="roa_dec")
years = sorted(fin.year.unique())

rows = []
for t in years:
    top = dec.index[dec[t] == 9]
    for k in range(1, 11):
        if t + k not in dec.columns: continue
        nxt = dec.loc[top, t + k]
        rows.append({"base": t, "k": k, "n": len(top), "still_top_decile": (nxt == 9).mean(),
                     "below_median": (nxt <= 4).mean(), "gone": nxt.isna().mean()})
pers = pd.DataFrame(rows)
pers["period"] = np.where(pers.base <= 2014, "base 2011-2014", "base 2015+")
pers_k = pers.groupby(["period", "k"])[["still_top_decile", "below_median", "gone", "n"]].mean().round(3)
tc.save_table(pers, "h3a_top_decile_persistence", index=False)

rev = []
for k in range(1, 11):
    for per, bases in [("base 2011-2014", [t for t in years if t <= 2014]), ("base 2015+", [t for t in years if t >= 2015])]:
        d = pd.concat([pd.DataFrame({"x": roa[t], "y": roa.get(t + k), "cik": roa.index, "t": t}) for t in bases if t + k in roa.columns]
                      ) if any(t + k in roa.columns for t in bases) else pd.DataFrame()
        if d.empty: continue
        d = d.dropna()
        d = d[d.x.between(d.x.quantile(0.01), d.x.quantile(0.99))]
        r = tc.fe_ols(d, "y", ["x"], [], "cik")
        rev.append({"period": per, "k": k, "slope": r.loc["x", "coef"], "se": r.loc["x", "se"], "n": int(r.loc["x", "n"])})
rev = pd.DataFrame(rev); tc.save_table(rev, "h3a_mean_reversion", index=False)
annual = rev[rev.k == 1].set_index("period").slope
print("one-year persistence slope (1 - annual reversion):", annual.round(3).to_dict())
pers_k.unstack(0).loc[[1, 3, 5, 10]] if 10 in pers_k.index.get_level_values(1) else pers_k

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for i, (per, g) in enumerate(pers.groupby("period")):
    m = g.groupby("k")[["still_top_decile", "below_median", "gone"]].mean() * 100
    axes[0].plot(m.index, m.still_top_decile, "o-", color=tc.C[i], ms=5, label=f"Still top decile ({per})")
    axes[0].plot(m.index, m.gone, "o--", color=tc.C[i], ms=4, lw=1.2, alpha=0.7, label=f"No longer filing ({per})")
axes[0].set_xlabel("Years later"); axes[0].set_ylabel("% of top-decile ROA firms"); axes[0].set_ylim(0, 100)
axes[0].set_title("Top-decile ROA firms, k years later", loc="left"); axes[0].legend(loc="upper right", fontsize=8)
for i, (per, g) in enumerate(rev.groupby("period")):
    axes[1].errorbar(g.k, g.slope, yerr=1.96 * g.se, fmt="o-", color=tc.C[i], ms=5, capsize=2, label=per)
axes[1].axhline(0, color=tc.INK, lw=0.6)
axes[1].set_xlabel("Years later"); axes[1].set_ylabel("Slope of ROA gap at t+k on gap at t")
axes[1].set_title("How much of a profitability gap survives", loc="left"); axes[1].legend(loc="upper right")
fig.tight_layout(); tc.save_fig(fig, "h3_fig1_roa_persistence")

## H3b · Do industry leaders stay leaders?

A leader is the largest firm by revenue in its 4-digit SIC industry, among industries with at least 3 SEC filers. Outcomes five years later:
* **still leader**
* **still filing but no longer leader**
* **gone**: no longer filing, through acquisition or failure.

Industries that drop below 3 filers are excluded from the base.

In [ ]:
fin["ind_year"] = fin.sic + "_" + fin.year.astype(str)
nfirm = fin.groupby("ind_year").revenue.transform("size")
fin["share"] = fin.revenue / fin.groupby("ind_year").revenue.transform("sum")
fin["rank"] = fin.groupby("ind_year").revenue.rank(ascending=False, method="first")
base = fin[nfirm >= 3]
lead = base[base["rank"] == 1][["cik", "sic", "year", "share", "entityName"]]
H = 5
rows = []
for _, r in lead.iterrows():
    t2 = r.year + H
    if t2 > fin.year.max(): continue
    ind_t2 = fin[(fin.sic == r.sic) & (fin.year == t2)]
    if len(ind_t2) < 3: continue
    me = ind_t2[ind_t2.cik == r.cik]
    status = "gone" if me.empty else ("still leader" if me["rank"].iloc[0] == 1 else "lost lead")
    rows.append({"cik": r.cik, "firm": r.entityName, "sic": r.sic, "base": r.year, "share": r.share,
                 "dominant": r.share >= 0.5, "status": status})
lp = pd.DataFrame(rows)
tc.save_table(lp, "h3b_leader_persistence", index=False)
by = (pd.crosstab([lp.base], lp.status, normalize="index") * 100).reindex(columns=["still leader", "lost lead", "gone"]).fillna(0)
by["n"] = lp.groupby("base").size()
dom = (pd.crosstab(lp.dominant, lp.status, normalize="index") * 100).reindex(columns=["still leader", "lost lead", "gone"]).round(1)
pre_post = (pd.crosstab(np.where(lp.base <= 2014, "base 2011-2014", "base 2015-2020"), lp.status, normalize="index") * 100).round(1)
print("By leader type (share >= 50% = dominant):"); print(dom.to_string()); print(); print(pre_post.to_string())
by.round(1)

In [ ]:
fig, ax = plt.subplots(figsize=(8.5, 4))
cols = {"still leader": tc.C[0], "lost lead": tc.C[3], "gone": tc.GREY}
bottom = np.zeros(len(by))
for s_ in ["still leader", "lost lead", "gone"]:
    ax.bar(by.index, by[s_], bottom=bottom, color=cols[s_], label=s_, width=0.75, edgecolor="white", linewidth=1)
    bottom += by[s_].values
for x, n in by.n.items(): ax.text(x, 101, str(n), ha="center", fontsize=7.5, color=tc.MUTED)
ax.set_ylim(0, 108); ax.set_ylabel("% of industry leaders"); ax.set_xlabel(f"Base year (leaders counted above bars); outcome {H} years later")
ax.set_xticks(by.index); ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.2), ncol=3)
ax.set_title("Industry leaders five years later", loc="left")
fig.tight_layout(); tc.save_fig(fig, "h3_fig2_leader_persistence")

## H3c · Are the giants durable?

The largest US firms by public float (the cleaned H1 panel, with successor links). Five-year top-20 survival counts a firm as surviving if it is in the top 20 in both *t* and *t*+5.

In [ ]:
pf = tc.public_float_panel(range(2011, 2026))
names = pf.sort_values("year").groupby("cik").entityName.last()
rank = pf.assign(r=pf.groupby("year").val.rank(ascending=False, method="first")).pivot_table(index="cik", columns="year", values="r")
END = int(pf.year.max())
top14 = rank.index[rank[2014] <= 20]
t20 = pd.DataFrame({"firm": names[top14].values, "rank_2014": rank.loc[top14, 2014].values,
                    f"rank_{END}": rank.loc[top14, END].values}, index=top14).sort_values("rank_2014")
t20["still_top20"] = t20[f"rank_{END}"] <= 20
tc.save_table(t20, "h3c_top20_2014", index=False)
surv = []
for t in range(2011, END - 4):
    a, b = rank.index[rank[t] <= 20], rank.index[rank[t + 5] <= 20]
    a10, b10 = rank.index[rank[t] <= 10], rank.index[rank[t + 5] <= 10]
    surv.append({"base": t, "top20_survive": len(a.intersection(b)) / 20, "top10_survive": len(a10.intersection(b10)) / 10})
surv = pd.DataFrame(surv); tc.save_table(surv, "h3c_top_survival", index=False)
print(f"Of the 2014 top 20, {t20.still_top20.sum()} are in the top 20 in {END}.")
print(surv.round(2).to_string(index=False))
t20

In [ ]:
fig, ax = plt.subplots(figsize=(9, 7))
fell = []
for _, r in t20.iterrows():
    r1 = r[f"rank_{END}"]
    col = tc.C[0] if r.still_top20 else tc.GREY
    ax.text(-0.04, r.rank_2014, r.firm.title()[:30], ha="right", va="center", fontsize=8)
    if r1 == r1 and r1 <= 40:
        ax.plot([0, 1], [r.rank_2014, r1], color=col, lw=1.6, marker="o", ms=5)
        ax.text(1.04, r1, f"{int(r1)}", ha="left", va="center", fontsize=8, color=tc.MUTED)
    else:
        ax.plot([0, 0.25], [r.rank_2014, r.rank_2014 + 3], color=col, lw=1.2, ls=":")
        fell.append(f"{r.firm.title()[:22]} ({'not filing' if r1 != r1 else int(r1)})")
new_in = rank.index[(rank[END] <= 20) & ~rank.index.isin(top14)]
entrants = [f"{names[c].title()[:24]} ({int(rank.loc[c, END])})" for c in rank.loc[new_in, END].sort_values().index]
ax.axhline(20.5, color=tc.INK, ls=":", lw=0.8)
ax.text(1.3, 1, f"New to the top 20 in {END}:\n" + "\n".join(entrants), va="top", fontsize=8, color=tc.INK)
if fell:
    ax.text(1.3, 25, f"Fell below rank 40:\n" + "\n".join(fell), va="top", fontsize=8, color=tc.MUTED)
ax.set_xlim(-0.95, 2.0); ax.set_ylim(41, 0); ax.set_xticks([0, 1]); ax.set_xticklabels(["2014", str(END)])
ax.set_yticks([]); ax.grid(False)
for sp in ["left", "bottom"]: ax.spines[sp].set_visible(False)
ax.set_title(f"The 20 largest US firms of 2014: rank by public float in {END}\n(blue = still top 20; dotted line = rank 20)", loc="left")
fig.tight_layout(); tc.save_fig(fig, "h3_fig3_top20")

## H3d · How far in the future did 2014 prices look?

For the 100 largest non-financial firms by 2014 public float, we compare their free cash flow over 2015–2025 with their 2014 value.

* **Payback (undiscounted)** = Σ FCF over 2015–2025 ÷ 2014 float.
* **PV payback** discounts each year's FCF at 8% back to mid-2014.
* **Terminal share** = 1 − PV payback: the part of the 2014 price that, on these numbers, had to come from cash flows after 2025.

Thiel predicts that tech firms carry a higher terminal share than mature firms. "Tech" means SIC 357x (computers), 3661–3679 (communications equipment and electronics), 48xx (communications) or 737x (software and services).

In [ ]:
cf = tc.cash_flows(range(2015, 2026))
v14 = tc.value_near(pf, 2014, (0, -1, 1))
vEND = tc.value_near(pf, END, (0, -1))
sic = tc.sec_sic()
sic_f = pd.Series({c: sic.get(c, sic.get(next((o for o, n in tc.SUCCESSOR.items() if tc._final(o) == c), -1), np.nan)) for c in v14.index})
nonfin = sic_f.dropna()
nonfin = nonfin[~nonfin.str[:2].astype(int).between(60, 67)]
big = v14.reindex(nonfin.index).dropna().nlargest(100)
r = 0.08
cfp = cf[cf.cik.isin(big.index)].copy()
cfp["disc"] = (1 + r) ** -(cfp.year - 2014)
agg = cfp.groupby("cik").agg(fcf_sum=("fcf", "sum"), pv_fcf=("fcf", lambda s: (s * cfp.loc[s.index, "disc"]).sum()), years=("year", "nunique"))
pb = pd.DataFrame({"firm": names.reindex(big.index), "sic": nonfin.reindex(big.index), "value_2014_bn": big / 1e9}).join(agg)
pb = pb[pb.years >= 8].copy()
pb["payback"] = pb.fcf_sum / (pb.value_2014_bn * 1e9)
pb["pv_payback"] = pb.pv_fcf / (pb.value_2014_bn * 1e9)
pb["terminal_share"] = 1 - pb.pv_payback
pb[f"value_{END}_bn"] = vEND.reindex(pb.index) / 1e9
pb["value_multiple"] = pb[f"value_{END}_bn"] / pb.value_2014_bn
s4 = pb.sic.astype(str)
pb["tech"] = s4.str[:3].eq("357") | s4.astype(int).between(3661, 3679) | s4.str[:2].eq("48") | s4.str[:3].eq("737")
pb = pb.sort_values("value_2014_bn", ascending=False)
tc.save_table(pb, "h3d_cashflow_payback")
grp = pb.groupby("tech")[["payback", "pv_payback", "terminal_share", "value_multiple"]].median().rename(index={True: "tech", False: "other"})
grp["n"] = pb.groupby("tech").size().rename(index={True: "tech", False: "other"})
print(f"{len(pb)} firms with at least 8 years of cash flows"); print(grp.round(2).to_string())
pb.head(25)[["firm", "value_2014_bn", "fcf_sum", "payback", "pv_payback", "terminal_share", "value_multiple", "tech"]].assign(
    fcf_sum=lambda d: (d.fcf_sum / 1e9).round(0)).round(2)

In [ ]:
top = pb.head(30).iloc[::-1]
fig, ax = plt.subplots(figsize=(8.5, 8))
ax.barh(range(len(top)), top.pv_payback * 100, color=[tc.C[0] if t else tc.GREY for t in top.tech], height=0.7)
ax.axvline(100, color=tc.INK, ls=":", lw=1)
ax.set_yticks(range(len(top))); ax.set_yticklabels([f.title()[:30] for f in top.firm], fontsize=8)
ax.set_xlabel("Present value of 2015–2025 free cash flow, % of 2014 public float (8% discount rate)")
ax.text(101, len(top) - 0.5, "2014 price fully repaid", fontsize=8, va="top")
ax.set_title("How much of the 2014 price did the next 11 years repay? (blue = tech)", loc="left")
fig.tight_layout(); tc.save_fig(fig, "h3_fig4_payback")

## Verdict inputs and check cell — send me this output

In [ ]:
def _f(x):
    try: return round(float(x), 3)
    except Exception: return None
k_tab = pers.groupby(["period", "k"])[["still_top_decile", "below_median", "gone"]].mean()
summary = {
    "n_firm_years": int(len(fin)),
    "top_decile": {f"{p}|k={k}": {c: _f(v) for c, v in row.items()} for (p, k), row in k_tab.iterrows() if k in (1, 3, 5, 10)},
    "reversion_slopes": {f"{r_.period}|k={r_.k}": [_f(r_.slope), _f(r_.se)] for r_ in rev.itertuples() if r_.k in (1, 3, 5, 10)},
    "leaders_by_type": dom.reset_index().astype({"dominant": str}).to_dict("records"),
    "leaders_pre_post": pre_post.reset_index().rename(columns={"row_0": "period"}).to_dict("records"),
    "leaders_n": int(len(lp)),
    "top20_2014_still": int(t20.still_top20.sum()), "end_year": END,
    "top20_new_entrants": entrants,
    "top20_2014": t20[["firm", "rank_2014", f"rank_{END}"]].to_dict("records"),
    "top_survival": surv.round(2).to_dict("records"),
    "payback_by_group": grp.round(3).reset_index().to_dict("records"),
    "payback_top10": pb.head(10)[["firm", "value_2014_bn", "pv_payback", "terminal_share", "value_multiple", "tech"]].round(2).to_dict("records"),
}
(tc.OUT / "h3_summary.json").write_text(json.dumps(summary, indent=1, default=str))
print(json.dumps(summary, indent=1, default=str))